# ULMS G4X data section by section
- This version does quality control on each section individually from the proseg segmentation
- It adds in the spatialdata framework for visualization and data storage purposes
- A01 and H04 were tonsil and D02 did not transfer properly, so these three sections were removed.
- High-grade sarcoma (not ULMS) patient removed: E04, F04, G04
- After quality control, these samples were then put into the resolvi_v3 notebook and integrated and annotated together in resolvi_v3_annotation
- The end of this notebook maps the annotations from resolvi_v3 back to the sections (spatially) and visualizes them.

# Set up

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import anndata as ad
import spatialdata as sd
import spatialdata_plot
from tifffile import imread

sc.settings.n_jobs = -1  # Use all available cores
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.grid'] = False # turn the grid off
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.interactive = False
plt.ioff()
sc.settings.autoshow = False
plt.rcParams['savefig.dpi'] = 300
sc.set_figure_params(dpi_save=300, facecolor='white')

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpasq

# version control
print("seaborn:", sns.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("spatialdata:", sd.__version__)
print("spatialdata_plot:", spatialdata_plot.__version__)
print("squidpy:", sq.__version__)

/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


seaborn: 0.13.2
pandas: 2.3.2
numpy: 2.2.6
scanpy: 1.11.4
anndata: 0.12.2
spatialdata: 0.5.0
spatialdata_plot: 0.2.13
squidpy: 1.6.5


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [2]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
print(PARENT_DIR)

OUTPUT_MASTER_DIR = jpasq.create_output_dir(PARENT_DIR, 'qc')
print(OUTPUT_MASTER_DIR)
os.chdir(OUTPUT_MASTER_DIR)
sc.settings.figdir = OUTPUT_MASTER_DIR

DATA_DIR = PARENT_DIR.parent.parent / 'G4X/G4X_raw'
print(DATA_DIR)

/oak/stanford/groups/longaker/ULMS/revision/G4X
Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc
/oak/stanford/groups/longaker/ULMS/revision/G4X/qc
/oak/stanford/groups/longaker/ULMS/G4X/G4X_raw


In [3]:
jpa_markers = jpasq.import_markers((PARENT_DIR / 'ref/jpa_g4x_breast_panel.csv'), output_type='dict')
jpa_markers = {key: value for key, value in jpa_markers.items() if key != 'Plasma_cell'} # JCHAIN and IGHG1 not in this segmentation run
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4]

{'Adipocyte': ['LPL', 'ADIPOQ'], 'B_cell': ['MS4A1', 'CD79A', 'CD19'], 'DC': ['CD1C', 'CLEC9A', 'HLA-DRA', 'GPR183'], 'EC': ['PECAM1', 'VWF', 'ACKR1', 'LYVE1'], 'Fibroblast': ['FAP', 'LUM', 'DPT', 'COL1A1', 'FN1', 'THY1', 'MGP', 'POSTN'], 'Macrophage': ['CD163', 'CD68', 'TLR2', 'CSF1R', 'IDO1'], 'Mast_cell': ['KIT'], 'Monocyte': ['FCGR3A', 'CD14', 'S100A9', 'S100A10', 'TLR4', 'WARS1', 'STAT1'], 'Neutrophil': ['CD33', 'CD44', 'CEACAM8', 'VEGFA'], 'NK_cell': ['GNLY', 'FCGR3A', 'NCAM1', 'KLRB1', 'KLRF1', 'KLRD1', 'NKG7', 'GZMA', 'GZMB', 'GZMH'], 'Pericyte': ['RGS5', 'PDGFRB', 'CSPG4'], 'Plasma_cell': ['CD38', 'SDC1', 'CD79A', 'MZB1', 'JCHAIN', 'TNFRSF17', 'IGHA1', 'IGHD', 'IGHG1', 'IGHM'], 'Proliferation': ['MKI67', 'TOP2A', 'RRM2'], 'Smooth_muscle': ['DES', 'ESR1', 'PGR', 'ACTA2', 'TAGLN', 'MYH11', 'ACTG2', 'AR', 'MYL9', 'MYLK', 'TNC'], 'T_cell': ['CD2', 'CD3D', 'CD3E', 'IL7R', 'CD247', 'CD4', 'CD8A']}


In [4]:
sections = ["A01","B01","C01","D01","E01","F01","G01","H01",
            "A02","B02","C02","D02","E02","F02","G02","H02",
            "A03","B03","C03","D03","E03","F03","G03","H03", 
            "A04","B04","C04","D04","E04","F04","G04","H04"]
sections_to_drop = ["A01", "H04", "D02", "E04", "F04", "G04"]
sections = [section for section in sections if section not in sections_to_drop]
sections

['B01',
 'C01',
 'D01',
 'E01',
 'F01',
 'G01',
 'H01',
 'A02',
 'B02',
 'C02',
 'E02',
 'F02',
 'G02',
 'H02',
 'A03',
 'B03',
 'C03',
 'D03',
 'E03',
 'F03',
 'G03',
 'H03',
 'A04',
 'B04',
 'C04',
 'D04']

In [5]:
len(sections)

26

# Section by section QC

## B01

In [7]:
section = 'B01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/B01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (70915, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (70915, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [9]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 68798 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


## C01

In [12]:
section = 'C01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/C01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (62991, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (62991, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [13]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 59599 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [14]:
# Removing spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=150, method='max', n_std=2)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))

Before filtering out spatial outliers: C01 59599


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


After filtering out spatial outliers: C01 59134


In [15]:
# Running spatial outliers with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=10, method='max', n_std=8)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))

Before filtering out spatial outliers: C01 59134
After filtering out spatial outliers: C01 59123


In [16]:
adata.write_h5ad(f'{section}_preprocessed.h5ad')

## D01

In [18]:
section = 'D01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/D01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (69634, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (69634, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [19]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 67404 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [20]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=7)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))

Before filtering out spatial outliers: D01 67404
After filtering out spatial outliers: D01 67362


In [21]:
adata.write_h5ad(f'{section}_preprocessed.h5ad')

## E01

In [22]:
section = 'E01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/E01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (46286, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (46286, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [23]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 43078 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [24]:
# Removing spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=10)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))

Before filtering out spatial outliers: E01 43078
After filtering out spatial outliers: E01 43057


In [25]:
# Removing spatial outliers again
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=5, method='max', n_std=10)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))

Before filtering out spatial outliers: E01 43057
After filtering out spatial outliers: E01 43036


In [26]:
adata.write_h5ad(f'{section}_preprocessed.h5ad')

## F01

In [27]:
section = 'F01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/F01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (59445, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (59445, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [28]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 56873 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [29]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=15)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: F01 56873
After filtering out spatial outliers: F01 56866


## G01

In [36]:
section = 'G01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/G01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (71096, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (71096, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [37]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 55879 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [38]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=20, method='max', n_std=12)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: G01 55879
After filtering out spatial outliers: G01 55864


## H01

In [39]:
section = 'H01'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H01
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H01
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L001_486a146fe865023d7/H01/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (172464, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (172464, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [40]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 160432 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [41]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=20, method='max', n_std=15)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: H01 160432
After filtering out spatial outliers: H01 160426


## A02

In [42]:
section = 'A02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/A02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (145646, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (145646, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [43]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 142992 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [44]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=12)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: A02 142992
After filtering out spatial outliers: A02 142979


## B02

In [45]:
section = 'B02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/B02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (70587, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (70587, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [46]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 68309 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [47]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=15)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: B02 68309
After filtering out spatial outliers: B02 68297


## C02

In [48]:
section = 'C02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/C02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (86016, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (86016, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [49]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 78765 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [50]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=15)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: C02 78765
After filtering out spatial outliers: C02 78759


## Skip D02 because it failed to transfer to the slide during the G4X run

## E02

In [51]:
section = 'E02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/E02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (136847, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (136847, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [52]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 133578 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [53]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=15)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: E02 133578
After filtering out spatial outliers: E02 133574


## F02

In [54]:
section = 'F02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/F02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (103407, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (103407, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [55]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 100676 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [56]:
# Removing spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=12)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: F02 100676
After filtering out spatial outliers: F02 100671


## G02

In [57]:
section = 'G02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/G02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (119420, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (119420, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [58]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 113140 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [59]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=15, method='max', n_std=9)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: G02 113140
After filtering out spatial outliers: G02 113097


## H02

In [60]:
section = 'H02'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H02
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H02
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/H02/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (140839, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (140839, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [61]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 120873 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [62]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=20)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: H02 120873
After filtering out spatial outliers: H02 120861


## A03

In [63]:
section = 'A03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/A03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (72375, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (72375, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [64]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 71317 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [65]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=20)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: A03 71317
After filtering out spatial outliers: A03 71310


## B03

In [66]:
section = 'B03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/B03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (60160, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (60160, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [67]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 59080 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [68]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=50, method='max', n_std=12)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: B03 59080
After filtering out spatial outliers: B03 59064


## C03

In [69]:
section = 'C03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/C03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (69855, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (69855, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [70]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 68946 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [71]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=40, method='max', n_std=6)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: C03 68946
After filtering out spatial outliers: C03 68819


## D03

In [72]:
section = 'D03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section,change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/D03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (71135, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (71135, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [73]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 69149 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [74]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=75, method='max', n_std=7)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: D03 69149
After filtering out spatial outliers: D03 68831


## E03

In [75]:
section = 'E03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/E03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/E03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (82529, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (82529, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [76]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 80746 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [77]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=35, method='max', n_std=12)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: E03 80746
After filtering out spatial outliers: E03 80684


In [78]:
# Remove spatial outliers again with more stringent cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=200, method='max', n_std=3)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: E03 80684
After filtering out spatial outliers: E03 79914


## F03

In [81]:
section = 'F03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/F03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/F03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (48489, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (48489, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [82]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 45978 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [83]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=2)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: F03 45978
After filtering out spatial outliers: F03 45899


In [84]:
# Remove spatial outliers again
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=15, method='max', n_std=8)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: F03 45899
After filtering out spatial outliers: F03 45860


## G03

In [85]:
section = 'G03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/G03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/G03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (21257, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (21257, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [86]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 19770 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [87]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=25, method='max', n_std=4)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: G03 19770
After filtering out spatial outliers: G03 19763


## H03

In [88]:
section = 'H03'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H03
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/H03
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/H03/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (17535, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (17535, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [89]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 15984 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [90]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=15, method='max', n_std=3)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: H03 15984
After filtering out spatial outliers: H03 15930


In [91]:
# Remove spatial outliers again
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=5)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: H03 15930
After filtering out spatial outliers: H03 15923


## A04

In [99]:
section = 'A04'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A04
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/A04
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L004_424ec4f5bf7d29d9e/A04/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (18655, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (18655, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [100]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 17763 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [101]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=160, method='max', n_std=3)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: A04 17763
After filtering out spatial outliers: A04 17603


In [102]:
# Remove spatial outliers again
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=20, method='max', n_std=4)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: A04 17603
After filtering out spatial outliers: A04 17584


## B04

In [103]:
section = 'B04'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B04
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/B04
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L004_424ec4f5bf7d29d9e/B04/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (85737, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (85737, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [104]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 83136 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [105]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=30, method='max', n_std=6)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: B04 83136
After filtering out spatial outliers: B04 82993


## C04

In [106]:
section = 'C04'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C04
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/C04
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L004_424ec4f5bf7d29d9e/C04/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (104686, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (104686, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [107]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 99867 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [108]:
# Running spatial outliers again with more stringent filtering cutoffs
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=75, method='max', n_std=5)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: C04 99867
After filtering out spatial outliers: C04 99701


## D04

In [109]:
section = 'D04'
output_dir = jpasq.create_output_dir(OUTPUT_MASTER_DIR, section, change_dir=True)
sdata = jpasq.load_g4x_zarr(DATA_DIR, section, proseg_folder_label='proseg_no_igj')
print(sdata)

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D04
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/qc/D04
INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
SpatialData object, with associated Zarr store: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L004_424ec4f5bf7d29d9e/D04/segmentation/proseg_no_igj/proseg-output.zarr
├── Images
│     └── 'hne': DataArray[cyx] (3, 19200, 15232)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 10) (3D points)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (89949, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (89949, 304)
with coordinate systems:
    ▸ 'global', with elements:
        hne (Images), transcripts (Points), cell_boundaries (Shapes)
with the following elements not in the Zarr store:
    ▸ hne (Images)


In [110]:
sdata = jpasq.g4x_qc(sdata, min_counts=20, min_genes=5)
adata = sdata.tables['table']
print(adata)
adata.write_h5ad(f'{section}_preprocessed.h5ad')

INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                


/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:128: UserWarning: Key `table` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/home/jpagolia/miniforge3/envs/jpa_squidpy/lib/python3.12/site-packages/spatialdata/_core/_elements.py:108: UserWarning: Key `cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
AnnData object with n_obs × n_vars = 85436 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes'
    var: 'gene', 'total_count', 'lambda_bg_0', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'proseg_run', 'section'
    obsm: 'spatial'


In [111]:
# Remove spatial outliers
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=300, method='max', n_std=4)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: D04 85436
After filtering out spatial outliers: D04 84174


In [112]:
# Remove spatial outliers a second time
print("Before filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.obs['spatial_outlier'] = jpasq.find_spatial_outliers(adata, n_neighbors=100, method='max', n_std=6)
adata = adata[~adata.obs['spatial_outlier']].copy()
print("After filtering out spatial outliers: " + section + ' ' + str(adata.n_obs))
adata.write_h5ad(f'{section}_preprocessed.h5ad')

Before filtering out spatial outliers: D04 84174
After filtering out spatial outliers: D04 83911


## Skip E04, F04, and G04 since this patient had high-grade cervical sarcoma, not leiomyosarcoma